In [ ]:
REPO_URL = "https://github.com/MberkKeskin/turkish_legal_rag_assistant.git"
PROJECT_DIR = "/content/turkish_legal_rag_assistant/FINAL_SUBMISSION"

HF_MODELS = {
    "bge-m3-legal-ft-system2": "Berk2003/bge-m3-legal-ft-system2",
    "bge_reranker_legal_ft_v4_error_mined_hf": "Berk2003/bge-reranker-legal-ft-v4-error-mined-hf",
    "qwen2_5_3b_legal_lora_sft_faithful_v2_final": "Berk2003/qwen2-5-3b-legal-lora-sft-faithful-v2-final",
}

print("Repository:", REPO_URL)
print("Project dir:", PROJECT_DIR)
print("Hugging Face models:")
for local_name, repo_id in HF_MODELS.items():
    print(f"  {local_name} <- {repo_id}")

!nvidia-smi || true

In [ ]:
%cd /content
!rm -rf turkish_legal_rag_assistant
!git clone "$REPO_URL"
%cd "$PROJECT_DIR"

!pwd
!ls

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -U huggingface_hub

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download
import shutil
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

models_dir = Path(PROJECT_DIR) / "models"

# GitHub'dan eksik/boş models klasörü geldiyse temizle
if models_dir.exists():
    shutil.rmtree(models_dir)

models_dir.mkdir(parents=True, exist_ok=True)

def count_files(path):
    path = Path(path)
    return len([x for x in path.rglob("*") if x.is_file()]) if path.exists() else 0

def folder_size_mb(path):
    path = Path(path)
    return sum(x.stat().st_size for x in path.rglob("*") if x.is_file()) / (1024 * 1024) if path.exists() else 0

for local_name, repo_id in HF_MODELS.items():
    target_dir = models_dir / local_name

    print("\n" + "=" * 90)
    print("Downloading model")
    print("HF repo   :", repo_id)
    print("Local path:", target_dir)
    print("=" * 90)

    snapshot_download(
        repo_id=repo_id,
        repo_type="model",
        local_dir=str(target_dir),
        local_dir_use_symlinks=False,
    )

    print("Downloaded:", local_name)
    print("Files:", count_files(target_dir))
    print("Size MB:", round(folder_size_mb(target_dir), 2))

print("\nFinal model check:")
for local_name in HF_MODELS:
    p = models_dir / local_name
    print(f"{local_name:60s} exists={p.exists()} files={count_files(p)} size_mb={round(folder_size_mb(p), 2)}")

missing = [
    name for name in HF_MODELS
    if not (models_dir / name).exists() or count_files(models_dir / name) == 0
]

if missing:
    raise RuntimeError(f"Missing model folders: {missing}")

print("\nAll Hugging Face models downloaded successfully.")

In [ ]:
from pathlib import Path

checks = {
    "models/bge-m3-legal-ft-system2": ["model.safetensors"],
    "models/bge_reranker_legal_ft_v4_error_mined_hf": ["model.safetensors"],
    "models/qwen2_5_3b_legal_lora_sft_faithful_v2_final": ["adapter_model.safetensors"],
}

def size_mb(path):
    return path.stat().st_size / (1024 * 1024)

for folder, required_files in checks.items():
    folder_path = Path(folder)
    print("\n" + "=" * 80)
    print(folder)
    print("exists:", folder_path.exists())

    for req in required_files:
        matches = list(folder_path.rglob(req)) if folder_path.exists() else []
        if not matches:
            raise RuntimeError(f"Missing required file: {folder}/{req}")

        for m in matches:
            print(req, "->", m, "| size MB:", round(size_mb(m), 2))

print("\nRequired model files are valid.")

In [ ]:
from pathlib import Path

required_items = [
    "README.md",
    "requirements.txt",
    "final_ui.py",
    "smoke_test.py",
    "smoke_test_light.py",
    "evaluate_custom_benchmark.py",
    "app",
    "data",
    "models",
]

print("Project file check:")
for item in required_items:
    p = Path(item)
    print(f"{item:40s}", "OK" if p.exists() else "MISSING")

In [ ]:
exec(open("final_ui.py", encoding="utf-8").read())